In [7]:
import pandas as pd
from datasets import Dataset

data = [
    {
        "raw_notice": "All final-year students must register for End-Semester Exams by Oct 10. Late fees apply. Faculty must verify lists by Oct 12.",
        "student_summary": "Register for End-Semester Exams on the portal by October 10 to avoid late fee penalties.",
        "faculty_summary": "Verify final-year student exam eligibility lists on the portal by October 12."
    },
    {
        "raw_notice": "Hostel fee payment window opens Sept 25. Final deadline is Oct 5. Warden office verification required by Oct 7.",
        "student_summary": "Pay hostel fees between September 25 and October 5.",
        "faculty_summary": "Warden office must complete hostel fee verification by October 7."
    }
]

# Transform into task-conditioned rows
formatted_rows = []
for item in data:
    # Row for Student Persona
    formatted_rows.append({
        "input_text": f"summarize for student: {item['raw_notice']}",
        "target_summary": item["student_summary"]
    })
    # Row for Faculty Persona
    formatted_rows.append({
        "input_text": f"summarize for faculty: {item['raw_notice']}",
        "target_summary": item["faculty_summary"]
    })

raw_dataset = Dataset.from_pandas(pd.DataFrame(formatted_rows))
print("Sample Dataset Row:")
print(raw_dataset[0])

Sample Dataset Row:
{'input_text': 'summarize for student: All final-year students must register for End-Semester Exams by Oct 10. Late fees apply. Faculty must verify lists by Oct 12.', 'target_summary': 'Register for End-Semester Exams on the portal by October 10 to avoid late fee penalties.'}


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Preprocessing Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target_summary"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize Dataset
tokenized_dataset = raw_dataset.map(preprocess_function, batched=True)

# Define Fine-Tuning Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results_bart",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

# Initialize Trainer & Train
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
)

print("Starting Fine-Tuning Loop...")
trainer.train()
print("Training Complete!")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Starting Fine-Tuning Loop...


c:\ProjectFiles\uni-comms-intelligence\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,15.344182
2,12.491250
3,11.734653
4,11.803929
5,11.212868
6,11.282722


Training Complete!


In [9]:
def generate_persona_summary(raw_notice: str, role: str = "student") -> str:
    input_prompt = f"summarize for {role}: {raw_notice}"
    inputs = tokenizer(input_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    summary_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=60,
        min_length=10,
        num_beams=4,
        no_repeat_ngram_size=3,        # Blocks repeated 3-word phrases
        repetition_penalty=1.2,        # Penalizes token repetition
        early_stopping=True
    )
    
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test User Inputs
test_notice = "CIRCULAR: Library books must be returned by September 30. Faculty approvals needed for extensions."

print("\n--- USER GENERATION TEST ---")
print("Student View:", generate_persona_summary(test_notice, role="student"))
print("Faculty View:", generate_persona_summary(test_notice, role="faculty"))


--- USER GENERATION TEST ---
Student View: CIRCULAR REQUIREMENTS: CIRC SPECIFIC NEEDED TO BE CONTINUEDLY: Items must be returned by September 30. Faculty approvals needed for extensions.
Faculty View: †C'E LIBRATAL: C RE-SELED GIVED FEELS FALL UP SUPPLY MUNGELED TO C SETS DESET FEFEL FEU FIXED FEED FE FE FE FEL FE FE RELIM PLAT


In [ ]:
import evaluate
import spacy

# 1. ROUGE Metrics Evaluation
rouge = evaluate.load("rouge")

predictions = [generate_persona_summary(row["input_text"].replace("summarize for student: ", ""), role="student") for row in raw_dataset]
references = [row["target_summary"] for row in raw_dataset]

rouge_results = rouge.compute(predictions=predictions, references=references)
print("\n--- EVALUATION RESULTS ---")
print("ROUGE Metrics:", rouge_results)

# 2. Entity Hallucination Check
nlp = spacy.load("en_core_web_sm")

def check_date_hallucinations(source_text: str, generated_summary: str) -> bool:
    source_doc = nlp(source_text)
    summary_doc = nlp(generated_summary)
    
    source_dates = {ent.text.lower() for ent in source_doc.ents if ent.label_ in ["DATE", "TIME"]}
    summary_dates = {ent.text.lower() for ent in summary_doc.ents if ent.label_ in ["DATE", "TIME"]}
    
    # Check if generated summary contains dates missing from the source text
    hallucinated_dates = summary_dates - source_dates
    has_hallucination = len(hallucinated_dates) > 0
    
    if has_hallucination:
        print(f"Hallucination Warning! Generated ungrounded dates: {hallucinated_dates}")
    else:
        print("Date Check Passed: Zero date hallucinations detected.")
        
    return has_hallucination

# Test Guardrail Check
check_date_hallucinations(test_notice, generate_persona_summary(test_notice, role="student"))


--- EVALUATION RESULTS ---
ROUGE Metrics: {'rouge1': np.float64(0.23582544302987657), 'rouge2': np.float64(0.07194127522527152), 'rougeL': np.float64(0.2149921096965432), 'rougeLsum': np.float64(0.2149921096965432)}
✅ Date Check Passed: Zero date hallucinations detected.


False